# Overhead

In [ ]:
import pandas as pd
import numpy as np
import re

variants = pd.read_csv("card_variants_03.csv")
prices = pd.read_csv("price_history_03.csv")

""" Do not truncate columns to help with cleaning """
pd.set_option("display.max_colwidth", None)

# View Data

## View Dataset Structure

In [ ]:
variants.shape, prices.shape

In [ ]:
variants.sample(3)

In [ ]:
prices.sample(3)

In [ ]:
milestones.sample(3)

In [ ]:
variants.columns

In [ ]:
prices.columns

## Inspect Columns

In [ ]:
prices["condition"].unique()

In [ ]:
prices["rarity"].unique()

# Prepare + Process Data


## Data Cleaning

### Data Cleaning Set up

#### Map Columns

In [ ]:
# maps to shorten and clean ids
set_map = {
    "romance-dawn-one-piece-card-game": "OP01",
    "paramount-war-one-piece-card-game": "OP02",
    "pillars-of-strength-one-piece-card-game": "OP03",
    "kingdoms-of-intrigue-one-piece-card-game": "OP04",
    "awakening-of-the-new-era-one-piece-card-game": "OP05",
    "wings-of-the-captain-one-piece-card-game": "OP06",
    "500-years-in-the-future-one-piece-card-game": "OP07",
    "two-legends-one-piece-card-game": "OP08",
    "emperors-in-the-new-world-one-piece-card-game": "OP09",
    "royal-blood-one-piece-card-game": "OP10",
    "a-fist-of-divine-speed-one-piece-card-game": "OP11",
    "legacy-of-the-master-one-piece-card-game": "OP12",
    "carrying-on-his-will-one-piece-card-game": "OP13",
    "the-azure-sea-s-seven-one-piece-card-game": "OP14",
    "extra-booster-memorial-collection-one-piece-card-game": "EB01",
    "extra-booster-anime-25th-collection-one-piece-card-game": "EB02",
    "extra-booster-one-piece-heroines-edition-one-piece-card-game": "EB03",
    "premium-booster-the-best-one-piece-card-game": "PRB01",
    "premium-booster-the-best-vol-2-one-piece-card-game": "PRB02",
}

rarity_map = {
    "Common": "c",
    "Uncommon": "uc",
    "Rare": "r",
    "Super Rare": "sr",
    "Secret Rare": "sec",
    "Leader": "l",
    "Don": "don",
    "DON!!": "don",
    "Promo": "p",
    "Treasure Rare": "tr",
}

condition_map = {
    "Near Mint": "nm",
    "Lightly Played": "lp",
    "Moderately Played": "mp",
    "Heavily Played": "hp",
    "Damaged": "d",
    "Sealed": "sealed",
}

# for reordering card variants columns
variant_cols_order = [
    "variant_id",
    "card_id",
    "card_number",
    "set_id",
    "set_name",
    "card_name",
    "rarity",
    "rarity_variant",
    "printing",
    "condition",
    "min_price_all_time",
    "max_price_all_time",
    "min_price_date",
    "max_price_date",
    "set_release_date",
    "price_history",
]

# for reordering price history columns
prices_cols_order = [
    "variant_id",
    "price",
    "date",
]

#### Helpers

In [ ]:
""" Lower case, strip, and replace spaces, underscores, and punctuation with dashes """
def slugify(s: str) -> str:
    if s is None:
        return None
    s = str(s).strip().lower()
    s = s.replace("&", "and")
    s = s.replace(".", "-")
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"[\s_]+", "-", s)
    s = re.sub(r"-{2,}", "-", s)
    return s.strip("-")

### General Cleaning

#### Fix Dates

##### Fix card_variants.csv dates

In [ ]:
date_cols = [
    "min_price_date",
    "max_price_date",
    "set_release_date",
]

for col in date_cols:
    variants[col] = pd.to_datetime(variants[col], errors="coerce")

for col in ["min_price_date", "max_price_date"]:
    variants[col] = variants[col].dt.date


##### Fix price_history.csv dates

In [ ]:
prices["datetime_utc"] = pd.to_datetime(
    prices["timestamp_utc"],
    errors="coerce",
    utc=True
)

mask = prices["datetime_utc"].isna()

prices.loc[mask, "datetime_utc"] = pd.to_datetime(
    prices.loc[mask, "timestamp_utc"],
    unit="s",
    errors="coerce",
    utc=True
)

prices["date"] = pd.to_datetime(prices["datetime_utc"], errors="coerce")

#### Remove Duplicates

##### Fix card_variants.csv Duplicates

In [ ]:
""" View card_variants.csv Duplicates """
dupes = variants[variants.duplicated(keep=False)]
dupes.sort_values(["variant_id"])

In [ ]:
""" Drop card_variants.csv Duplicates """
variants_before = len(variants)

variants = (
    variants
    .sort_values("last_updated_utc", ascending=False)
    .drop_duplicates(
        subset=["variant_id"],
        keep="first"
    )
)

variants_after = len(variants)

##### Fix price_history.csv Duplicates

In [ ]:
""" View price_history.csv Duplicates """
dupes = prices[prices.duplicated(keep=False)]
dupes.sort_values(["variant_id"])

In [ ]:
""" Drop price_history.csv Duplicates """
prices_before = len(prices)

prices = (
    prices
    .sort_values(
        ["variant_id", "timestamp_utc"], ascending=False
    )
    .drop_duplicates(
        subset=["variant_id","timestamp_utc"],
        keep="last"
    )
)

prices_after = len(prices)

prices = prices.sort_values(
    ["variant_id", "timestamp_utc"]
)

print(f"Dropped {variants_before - variants_after} rows from card_variants.csv | Total Rows Before: {variants_before} | Total Rows After: {variants_after}")
print(f"Dropped {prices_before - prices_after} rows from price_history.csv | Total Rows Before: {prices_before} | Total Rows After: {prices_after}")

#### Address Un-aligned Variant IDs

##### Inspect unaligned variant ids

In [ ]:
""" Check for unaligned variant ids """
prices_ids = set(prices["variant_id"].dropna())
variant_ids = set(variants["variant_id"].dropna())

missing_in_variants = prices_ids - variant_ids
missing_in_prices   = variant_ids - prices_ids

len(missing_in_variants), len(missing_in_prices)

In [ ]:
""" View unaligned nm & sealed variants ids """
missing_variants = variants[variants["variant_id"].isin(missing_in_prices)]

missing_variants.loc[
    missing_variants["condition"].isin(["Near Mint", "Sealed"]),
    [
        "variant_id",
        "card_name",
        "set_id",
        "printing",
        "language",
    ]
]

##### Flag unaligned ids as having no price history

In [ ]:
variants["price_history"] = variants["variant_id"].isin(prices["variant_id"])

### Clean card_variants.csv

#### Shorten IDs

In [ ]:
""" Create Temporary Placeholder Columns """
variants["set_code"] = variants["set_id"].map(set_map)
variants["rarity_code"] = variants["rarity"].map(rarity_map)
variants["condition_code"] = variants["condition"].map(condition_map)
variants["card_name_slug"] = variants["card_name"].apply(slugify)
variants["is_foil"] = variants["printing"].str.lower().eq("foil")

In [ ]:
""" Create Shortened IDs """
variants["short_card_id"] = np.where(
    variants["condition"].str.lower() =="sealed",
    variants["set_code"].str.lower()
    + "-" + variants["card_name_slug"],
    variants["set_code"].str.lower()
    + "-" + variants["card_name_slug"]
    + "-" + variants["rarity_code"]
)

variants["short_variant_id"] = (
    variants["short_card_id"]
    + "-" + variants["condition_code"]
    + variants["is_foil"].map({True: "-foil", False: ""})
)

#### Drop Columns and Re-name IDs

In [ ]:
variants = variants.drop(
    columns=[
        "game", 
        "current_price",
        "price_change_24hr",
        "set_id", 
        "card_id", 
        "variant_id", 
        "last_updated_utc", 
        "card_name_slug", 
        "rarity_code", 
        "condition_code",
        "tcgplayer_sku_id",
        "tcgplayer_id",
        "is_foil",
        "language",
        ], 
        errors="ignore")

variants = variants.rename(columns={
    "set_code": "set_id",
    "short_card_id": "card_id",
    "short_variant_id": "variant_id",
})

### Clean price_history.csv

#### Shorten IDs

In [ ]:
""" Create Temporary Placeholder Columns """
prices["set_code"] = prices["set_id"].map(set_map)
prices["rarity_code"] = prices["rarity"].map(rarity_map)
prices["condition_code"] = prices["condition"].map(condition_map)
prices["card_name_slug"] = prices["card_name"].apply(slugify)
prices["is_foil"] = prices["printing"].str.lower().eq("foil")

In [ ]:
""" Create Shortened IDs """
prices["short_card_id"] = np.where(
    prices["condition"].str.lower() =="sealed",
    prices["set_code"].str.lower()
    + "-" + prices["card_name_slug"],
    prices["set_code"].str.lower()
    + "-" + prices["card_name_slug"]
    + "-" + prices["rarity_code"]
)

prices["short_variant_id"] = (
    prices["short_card_id"]
    + "-" + prices["condition_code"]
    + prices["is_foil"].map({True: "-foil", False: ""})
)


#### Drop Columns and Re-name IDs

In [ ]:
prices = prices.drop(
    columns=[
    "game", 
    "timestamp_utc", 
    "datetime_utc", 
    "set_id", 
    "card_id", 
    "variant_id", 
    "card_name_slug", 
    "rarity_code", 
    "condition_code",
    "set_name",
    "card_name",
    "card_number",
    "rarity",
    "condition",
    "printing",
    "language",
    "set_id",
    "is_foil",
    "card_id",
    "set_code",
    "short_card_id",
    ], 
    errors="ignore"
)

prices = prices.rename(columns={
    "short_variant_id": "variant_id",
})

## Adding Features

### Add Days Since Release Column in Price History Table

In [ ]:
prices["date"] = pd.to_datetime(prices["date"], errors="coerce")

prices = prices.merge(
    variants[["variant_id", "set_release_date"]],
    on="variant_id",
    how="left"
)
prices["days_since_release"] = (
    prices["date"] - prices["set_release_date"]
).dt.days

prices = prices.drop(columns="set_release_date", errors="ignore")

prices["date"] = prices["date"].dt.date
variants["set_release_date"] = variants["set_release_date"].dt.date

### Create Rarity Variant Column


In [ ]:
v = variants.copy()

v["variant_lc"] = v["variant_id"].str.lower()
v["card_lc"] = v["card_id"].str.lower()
v["rarity_variant"] = v["printing"]


"""Sealed Product"""
v.loc[
    v["variant_lc"].str.contains("booster-box-case", na=False),
    "rarity_variant"
] = "Case"

v.loc[
    v["variant_lc"].str.contains("booster-box", na=False) &
    ~v["variant_lc"].str.contains("booster-box-case", na=False),
    "rarity_variant"
] = "Booster Box"

v.loc[
    v["variant_lc"].str.contains("sleeved-booster-pack", na=False),
    "rarity_variant"
] = "Blister"

v.loc[
    v["variant_lc"].str.contains("booster-pack", na=False) &
    ~v["variant_lc"].str.contains("sleeved-booster-pack", na=False),
    "rarity_variant"
] = "Booster Pack"


"""Don Cards"""
is_don = v["card_lc"].str.contains("don", na=False)


"""SP Variants"""
silver_sp_exceptions = [
    "op11-monkey-d-luffy-119-sp",
    "op14-buggy-op09-051-sp"
] # OP11 Luffy and Buggy Silver SPs were incorrectly identified as regular SPs

silver_sp_pattern = "|".join(silver_sp_exceptions)

v.loc[
    v["variant_lc"].str.contains("sp-gold", na=False),
    "rarity_variant"
] = "Gold SP"

v.loc[
    (
        v["variant_lc"].str.contains("sp-silver", na=False) |
        v["variant_lc"].str.contains(silver_sp_pattern, na=False, regex=True)
    ) &
    ~v["variant_lc"].str.contains("sp-gold", na=False),
    "rarity_variant"
] = "Silver SP"

v.loc[
    v["variant_lc"].str.contains("-sp-", na=False) &
    ~v["variant_lc"].str.contains("sp-gold|sp-silver", na=False) &
    ~v["variant_lc"].str.contains(silver_sp_pattern, na=False, regex=True),
    "rarity_variant"
] = "SP"


"""Wanted Posters"""
v.loc[
    v["variant_lc"].str.contains("wanted-poster", na=False),
    "rarity_variant"
] = "Wanted Poster"


"""Alternate Arts / Parallels"""
v.loc[
    ~is_don &
    ~v["variant_lc"].str.contains("super-alternate-art", na=False) &
    (   
        v["variant_lc"].str.contains("alternate-art", na=False) |
        v["variant_lc"].str.contains("parallel", na=False)
    ),
    "rarity_variant"
] = "Alternate Art"


"""Other Alts"""
v.loc[
    v["variant_lc"].str.contains("full-art", na=False),
    "rarity_variant"
] = "Full Art"

v.loc[
    v["variant_lc"].str.contains("textured-foil", na=False),
    "rarity_variant"
] = "Textured Foil"


"""Gold Don"""
v.loc[
    is_don &
    v["variant_lc"].str.contains("gold-don", na=False),
    "rarity_variant"
] = "Gold Don"


"""Red Mangas"""
v.loc[
    ~is_don &
    v["variant_lc"].str.contains("red-super-alternate-art", na=False),
    "rarity_variant"
] = "Red Manga"

"""Regular Mangas"""
v.loc[
    ~is_don & 
    ~v["variant_lc"].str.contains("red-super-alternate-art", na=False) &
    (
        v["variant_lc"].str.contains("super-alternate-art", na=False) |
         v["variant_lc"].str.contains("manga", na=False)
    ),
    "rarity_variant"
] = "Manga"


variants = v
variants = variants.drop(columns=["variant_lc", "card_lc"], errors="ignore")

# Export CSVs

## Reorder Columns

### Reorder card variants columns

In [ ]:
variants = variants[
    [c for c in variant_cols_order if c in variants.columns]
    + [c for c in variants.columns if c not in variant_cols_order]
]

### Reorder price history columns

In [ ]:
prices = prices[
    [c for c in prices_cols_order if c in prices.columns]
    + [c for c in prices.columns if c not in prices_cols_order]
]

In [ ]:
prices["date"].nunique()

### View Column Order

In [ ]:
variants.sample(20)

## Sort Tables & Export

In [ ]:
variants = variants.sort_values(["set_release_date", "card_number"])
prices = prices.sort_values(["variant_id", "date"])

VARIANTS_OUT = "variants_clean_16_3.csv"
PRICES_OUT = "prices_clean_16_3.csv"

variants.to_csv(
    VARIANTS_OUT,
    index=False
)

prices.to_csv(
    PRICES_OUT,
    index=False
)

print(f"Exported {VARIANTS_OUT} ({len(variants):,} rows)")
print(f"Exported {PRICES_OUT} ({len(prices):,} rows)")

# Building more features

## Set up

In [ ]:
variants = pd.read_csv("variants_clean_16_3.csv")
prices = pd.read_csv("prices_clean_16_3.csv")

## Create New Columns and Milestones

### Create Price Index Column

In [ ]:
"""Create Price Index column"""
def baseline_first_week(x):
    week = (
        x.loc[
            (x["days_since_release"] >= 0) &
            (x["days_since_release"] <= 7),
            "price"
        ]
        .dropna()
        .sort_values()
    )

    if len(week) == 0:
        return np.nan
    
    if len(week) >= 3:
        week = week.iloc[int(len(week) * 0.4):]

    return week.median()

baseline_week = (
    prices.sort_values(["variant_id", "date"])
    .groupby("variant_id", group_keys=False)[["days_since_release", "price"]]
    .apply(baseline_first_week)
    .rename("baseline_week")
    .reset_index()
)

"""Fallback 1: median of first 3 observations if no first-week data"""
post_release_prices = prices.loc[prices["days_since_release"] >= 0].copy()

fallback_first_3_rows = (
    post_release_prices.sort_values(["variant_id", "date"])
    .groupby("variant_id")
    .head(3)
)

baseline_first_3 = (
    fallback_first_3_rows
    .groupby("variant_id")["price"]
    .median()
    .rename("baseline_first_3")
    .reset_index()
)

"""Fallback 2: first available price"""
baseline_first = (
    post_release_prices.sort_values(["variant_id", "date"])
    .groupby("variant_id")["price"]
    .first()
    .rename("baseline_first")
    .reset_index()
)

"""Combine all baselines"""
baseline = baseline_week.merge(baseline_first_3, on="variant_id", how="outer")
baseline = baseline.merge(baseline_first, on="variant_id", how="outer")

baseline["baseline_price"] = (
    baseline["baseline_week"]
    .combine_first(baseline["baseline_first_3"])
    .combine_first(baseline["baseline_first"])
)

baseline["baseline_price"] = pd.to_numeric(baseline["baseline_price"], errors="coerce")

"""Track which baseline type was used"""
baseline["baseline_type"] = "week"

baseline.loc[
    baseline["baseline_week"].isna(),
    "baseline_type"
] = "first_3_observations"

baseline.loc[
    baseline["baseline_week"].isna() & baseline["baseline_first_3"].isna(),
    "baseline_type"
] = "first_price"


"""PRICE INDEX CALC"""
prices = prices.merge(
    baseline[["variant_id", "baseline_price", "baseline_type"]],
    on="variant_id",
    how="left"
)

prices["baseline_price"] = pd.to_numeric(prices["baseline_price"], errors="coerce")

post_release_mask = prices["days_since_release"] >= 0

prices.loc[post_release_mask, "price_index"] = (
    prices.loc[post_release_mask, "price"] / 
    prices.loc[post_release_mask, "baseline_price"] * 100
)

### Create Daily Return Column

In [ ]:
"""Create Daily Return column"""
prices = prices.sort_values(["variant_id", "date"])

prices["daily_return"] = (
    prices.groupby("variant_id")["price"]
    .pct_change()
)

prices["7d_return"] = (
    prices.groupby("variant_id")["price"]
    .pct_change(7)
)

### Create Max Days Since Release Column

In [ ]:
"""Create Max Days Since Release Column"""
max_days = (
    prices.groupby("variant_id")["days_since_release"]
    .max()
    .rename("max_days_available")
    .reset_index()
)
prices = prices.merge(max_days, on="variant_id", how="left")

### Create Observation Day Column

In [ ]:
"""Create Obersvation Day Column"""
prices["obs_day"] = (
    prices.groupby("variant_id")
    .cumcount()
)

### Create Lifecycle Stage Column

In [ ]:
"""Create Card Lifecycle Buckets"""
prices["lifecycle_stage"] = pd.cut(
    prices["days_since_release"],
    bins=(-7, 0, 7, 30, 90, 180, 365, 9999),
    labels=[
        "Pre-release (<0)",
        "Launch Week (0-6)",
        "Early Post-Launch (7-29)",
        "Early Phase (30-89)",
        "Mid Phase (90-179)",
        "Late Phase (180-364)",
        "Mature Phase (365+)",
    ],
    right=False
)

### Create Milestones

#### Setup Milestones

In [ ]:
window_stage = (
    prices
    .groupby(["variant_id", "lifecycle_stage"], observed=True)["price"]
    .median()
    .reset_index(name="stage_price")
)

milestones = (
    window_stage
    .pivot(index="variant_id",
           columns="lifecycle_stage",
           values="stage_price")
    .reset_index()
)

milestones = milestones.rename(columns={
        "Pre-release (<0)": "price_pre",
        "Launch Week (0-6)": "price_week",
        "Early Post-Launch (7-29)": "price_month",
        "Early Phase (30-89)": "price_early",
        "Mid Phase (90-179)": "price_mid",
        "Late Phase (180-364)": "price_late",
        "Mature Phase (365+)": "price_mature",
})

#### Calculate Compression Metrics

In [ ]:
milestones["change_pre_to_week_pct"] = (
    (milestones["price_week"] - milestones["price_pre"])
    / milestones["price_pre"]
) * 100
milestones.loc[
    (milestones["price_pre"].isna()) | (milestones["price_pre"] == 0),
    "change_pre_to_week_pct"
] = pd.NA

milestones["change_pre_to_month_pct"] = (
    (milestones["price_month"] - milestones["price_pre"])
    / milestones["price_pre"]
) * 100
milestones.loc[
    (milestones["price_pre"].isna()) | (milestones["price_pre"] == 0),
    "change_pre_to_month_pct"
] = pd.NA

milestones["change_week_to_month_pct"] = (
    (milestones["price_month"] - milestones["price_week"])
    /milestones["price_week"]
) * 100
milestones.loc[
    (milestones["price_week"].isna()) | (milestones["price_week"] == 0),
    "change_week_to_month_pct"
] = pd.NA

milestones["week_efficiency_ratio"] = (
    milestones["change_pre_to_week_pct"]
    /milestones["change_pre_to_month_pct"]
)
milestones.loc[
    (milestones["change_pre_to_month_pct"].isna()) |
    (milestones["change_pre_to_month_pct"] == 0),
    "week_efficiency_ratio"
] = pd.NA


milestones = milestones.merge(
    variants[["variant_id", "set_id", "rarity_variant"]],
    on="variant_id",
    how="left",
)


#### Calculate Pre-Release Metrics

In [ ]:
pre_release_summary = (
    prices.loc[prices["days_since_release"] < 0]
    .groupby("variant_id")["price"]
    .median()
    .rename("pre_release_price")
    .reset_index()
)

launch_week_summary = (
    prices.loc[
        (prices["days_since_release"] >= 0) &
        (prices["days_since_release"] <= 7)
    ]
    .groupby("variant_id")["price"]
    .median()
    .rename("launch_week_price")
    .reset_index()
)

milestones = milestones.merge(
    pre_release_summary,
    on="variant_id",
    how="left"
)

milestones = milestones.merge(
    launch_week_summary,
    on="variant_id",
    how="left"
)

milestones["pre_to_launch_week_pct"] = (
    (milestones["launch_week_price"] - milestones["pre_release_price"])
    / milestones["pre_release_price"]
) * 100

milestones.loc[
    (milestones["pre_release_price"].isna()) |
    (milestones["pre_release_price"] == 0),
    "pre_to_launch_week_pct"
] = pd.NA


## Reorder, Sort, and Export

### Standardize Table Order

In [ ]:
prices_cols_order = [
    "variant_id",
    "date",
    "obs_day",
    "max_days_available",
    "price",
    "baseline_price",
    "baseline_type"
    "price_index",
    "daily_return",
    "7d_return",
    "days_since_release",
    "lifecycle_stage",
    "source",
]

variant_cols_order = [
    "variant_id",
    "card_id",
    "card_number",
    "set_id",
    "set_name",
    "card_name",
    "rarity",
    "rarity_variant",
    "printing",
    "condition",
    "min_price",
    "max_price",
    "min_price_date",
    "max_price_date",
    "set_release_date",
    "price_history",
    "source",
]


milestones_cols_order = [
    "variant_id",
    "set_id",
    "rarity_variant",
    "price_pre",
    "price_week",
    "price_month",
    "price_early",
    "price_mid",
    "price_late",
    "price_mature",
    "change_pre_to_week_pct",
    "change_pre_to_month_pct",
    "change_week_to_month_pct",
    "week_efficiency_ratio",				
]

### Reorder Table Columns

In [ ]:
variants = variants[
    [c for c in variant_cols_order if c in variants.columns]
    + [c for c in variants.columns if c not in variant_cols_order]
]

prices = prices[
    [c for c in prices_cols_order if c in prices.columns]
    + [c for c in prices.columns if c not in prices_cols_order]
]

milestones = milestones[
    [c for c in milestones_cols_order if c in milestones.columns]
    + [c for c in milestones.columns if c not in milestones_cols_order]
]

### Sort and Export

In [ ]:
variants = variants.sort_values(["set_release_date", "card_number"])
prices = prices.sort_values(["variant_id", "date"])
milestones = milestones.sort_values(["variant_id"])

VARIANTS_OUT = "variants_clean_17_3.csv"
PRICES_OUT = "prices_clean_17_3.csv"
MILESTONES_OUT = "milestones.csv"

variants.to_csv(
    VARIANTS_OUT,
    index=False
)

prices.to_csv(
    PRICES_OUT,
    index=False
)

milestones.to_csv(
    MILESTONES_OUT,
    index=False
)


print(f"Exported {VARIANTS_OUT} ({len(variants):,} rows)")
print(f"Exported {PRICES_OUT} ({len(prices):,} rows)")
print(f"Exported {MILESTONES_OUT} ({len(milestones):,} rows)")

# Append CSVs

In [ ]:
"""Change These Variables"""
variants_1 = pd.read_csv("card_variants_03_04_07.csv")
variants_2 = pd.read_csv("card_variants_08.csv")

prices_1 = pd.read_csv("price_history_03_04_07.csv")
prices_2 = pd.read_csv("price_history_08.csv")

source_number_1 = "030407"
source_number_2 = "08"


"""No Change Variables"""
variants_1["source"] = source_number_1
variants_2["source"] = source_number_2

prices_1["source"] = source_number_1
prices_2["source"] = source_number_2

variants_all = pd.concat([variants_1, variants_2], ignore_index=True)

variants_clean = (
    variants_all.sort_values("source", ascending=False)
        .drop_duplicates(subset=["variant_id"], keep="first")
        .reset_index(drop=True)
)

prices_all = pd.concat([prices_1, prices_2], ignore_index=True)

prices_all["date"] = pd.to_datetime(prices_all["date"], errors="coerce").dt.normalize()

prices_clean = (
    prices_all.sort_values(["variant_id", "date", "source"])
        .drop_duplicates(subset=["variant_id", "date"], keep="last")
        .reset_index(drop=True)
)


"""Rename the file names here"""
variants_clean.to_csv("card_variants_09.csv", index=False)
prices_clean.to_csv("price_history_09.csv", index=False)